# Credit Card Spending Prediction – PAC Report
### Early Approaches & Experiments

**Analyst:** Darren Lin 
**Date:** November 24, 2025

## 1. Project Overview
This report documents my early modeling attempts and alternative pipelines from the Predictive Analytics Competition (PAC). The primary objective was to predict each customer's monthly credit card spending using a tabular dataset, evaluated by minimizing Root Mean Squared Error (RMSE). This notebook demonstrates initial trials with manual imputation, one-hot encoding by hand, basic tree models, and linear blending, revealing pitfalls of non-pipeline workflows.

## 2. Data Loading
Below I loaded training and scoring data and created my initial feature lists, without any engineered features.

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('analysis_data.csv')
scoring_df = pd.read_csv('scoring_data.csv')

target = 'monthly_spend'
features = [c for c in train_df.columns if c not in ['customer_id', target]]

## 3. Manual Imputation
Missing values were handled manually using median for numerical columns and mode for categorical columns.

In [2]:
from sklearn.impute import SimpleImputer

numeric_cols = train_df[features].select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in features if c not in numeric_cols]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

train_num = pd.DataFrame(num_imputer.fit_transform(train_df[numeric_cols]), columns=numeric_cols)
train_cat = pd.DataFrame(cat_imputer.fit_transform(train_df[categorical_cols]), columns=categorical_cols)
scoring_num = pd.DataFrame(num_imputer.transform(scoring_df[numeric_cols]), columns=numeric_cols)
scoring_cat = pd.DataFrame(cat_imputer.transform(scoring_df[categorical_cols]), columns=categorical_cols)

(This process was error-prone and fragile when new categories appeared in test data)

## 4. One-Hot Encoding
Categorical variables were manually one-hot encoded.
Columns might misalign if categories differed between train and scoring datasets.

In [3]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
train_cat_encoded = pd.DataFrame(encoder.fit_transform(train_cat), columns=encoder.get_feature_names_out(categorical_cols))
scoring_cat_encoded = pd.DataFrame(encoder.transform(scoring_cat), columns=encoder.get_feature_names_out(categorical_cols))

(Manual encoding required careful column alignment and was less robust than automated pipelines.)

## 5. Feature Matrix Construction
Numerical and encoded categorical features were concatenated for modeling.

In [4]:
X_train_full = pd.concat([train_num, train_cat_encoded], axis=1)
y_train_full = train_df[target]
X_score = pd.concat([scoring_num, scoring_cat_encoded], axis=1)

## 6. Train/Validation Split
A simple train-test split was used for validation, which can produce volatile RMSE depending on the random state.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

## 7. Training Basic Models
Random Forest and Gradient Boosting regressors were trained and evaluated separately.

In [6]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=7, random_state=42, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.07, max_depth=6, subsample=0.7, random_state=42)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)
val_rf_pred = rf.predict(X_val)
val_gb_pred = gb.predict(X_val)

(Parameter choices were not optimized and feature engineering was minimal.)

## 8. Manual Stacking
Stacking was implemented by blending RF and GB predictions with a LinearRegression meta-model.

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

val_stack = np.vstack([val_rf_pred, val_gb_pred]).T
meta = LinearRegression()
meta.fit(val_stack, y_val)
val_pred = meta.predict(val_stack)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
print(f'Stacking Validation RMSE: {rmse:.4f}')

Stacking Validation RMSE: 257.2487


(LinearRegression meta-model had issues with scale and no regularization; automated stacking in my final pipeline worked much better.)

## 9. Prediction and Submission File
Predictions were made using this stacked approach and saved in CSV format for submission.

In [10]:
score_rf_pred = rf.predict(X_score)
score_gb_pred = gb.predict(X_score)
score_stack = np.vstack([score_rf_pred, score_gb_pred]).T
score_pred = meta.predict(score_stack)
submission = pd.DataFrame({
    'customer_id': scoring_df['customer_id'],
    'monthly_spend': score_pred
})
submission.to_csv('submission9_other.csv', index=False)
print('Saved: submission9_other.csv')

Saved: submission9_other.csv


**Key Lessons:**
- Manual preprocessing was fragile, lacked transparency, and risked errors as features changed.

- Validation using a single split was not reliable; k-fold cross-validation would later reveal variable performance.

- A simple meta-model without regularization or scaling underperformed compared to more robust pipeline stacking.

- Automated pipelines and domain-driven feature engineering gave cleaner, reproducible, and higher-performing results.

**Improvements for Next Time:**
- Build preprocessing and modeling pipelines from the start to ensure robustness and flexibility.

- Prioritize systematic cross-validation instead of relying on random splits for evaluation.

- Explore regularized meta-models and proper feature scaling for ensemble blending.

- Invest more time in early, domain-inspired feature engineering rather than ad-hoc manual steps.

Thank You.